# 02 — Data Quality
Online Retail II

Audit source overlap, duplicated records, invoice identity, and transaction quality. Source rows are preserved.

## 1. Load and identify records

Load both worksheets, retain their original row positions, and confirm record counts.


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
source_columns = list(next(iter(sheets.values())).columns)

raw = pd.concat(
    [sheet.assign(source_sheet=name, source_row=np.arange(len(sheet)))
     for name, sheet in sheets.items()],
    ignore_index=True,
)

df = raw.copy()
df["invoice_id"] = df["Invoice"].astype("string").str.strip()
df["stock_code"] = df["StockCode"].astype("string").str.strip()
df["customer_id"] = df["Customer ID"].astype("Int64").astype("string")
df["invoice_timestamp"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["line_amount"] = df["Quantity"] * df["Price"]
df["source_record_key"] = df["source_sheet"] + ":" + df["source_row"].astype(str)

assert df["source_record_key"].is_unique
assert len(df) == sum(len(sheet) for sheet in sheets.values())
display(df.groupby("source_sheet").size().rename("rows").to_frame())
print("Total source rows:", len(df))

## 2. Worksheet overlap and duplicate records

Identify identical rows within and across worksheets, and quantify overlapping invoice IDs.


In [ ]:
# Compare source-column values, excluding the worksheet provenance columns.
# Hash matches are duplicate *candidates*; no rows are removed here.
df["row_fingerprint"] = pd.util.hash_pandas_object(
    df[source_columns], index=False
)
df["within_sheet_duplicate"] = df.duplicated(
    ["source_sheet", "row_fingerprint"], keep="first"
)

# Pair the first, second, etc. occurrence of each identical row across sheets.
df["fingerprint_occurrence"] = df.groupby(
    ["source_sheet", "row_fingerprint"], sort=False
).cumcount()
df["cross_sheet_duplicate_candidate"] = df.duplicated(
    ["row_fingerprint", "fingerprint_occurrence"], keep=False
)

invoice_sheets = df.groupby("invoice_id")["source_sheet"].nunique()
overlap_invoice_ids = invoice_sheets.index[invoice_sheets.gt(1)]

checks = pd.Series({
    "Source lines": len(df),
    "Within-sheet repeated lines beyond first": int(df["within_sheet_duplicate"].sum()),
    "Cross-sheet exact-row candidate pairs": int(df["cross_sheet_duplicate_candidate"].sum() // 2),
    "Invoice IDs appearing in both sheets": len(overlap_invoice_ids),
    "Rows linked to shared invoice IDs": int(df["invoice_id"].isin(overlap_invoice_ids).sum()),
})
display(checks.to_frame("count"))

display(df.loc[
    df["cross_sheet_duplicate_candidate"],
    ["source_sheet", "Invoice", "StockCode", "InvoiceDate", "Customer ID", "Quantity", "Price"]
].head(12))

## 3. Invoice identity checks

Check whether invoice IDs shared by the worksheets have conflicting transaction details.


In [ ]:
invoice_check = df.groupby("invoice_id", dropna=False).agg(
    source_sheets=("source_sheet", "nunique"),
    distinct_timestamps=("invoice_timestamp", "nunique"),
    distinct_customers=("customer_id", "nunique"),
    distinct_countries=("Country", "nunique"),
    first_timestamp=("invoice_timestamp", "min"),
    last_timestamp=("invoice_timestamp", "max"),
    line_count=("Invoice", "size"),
)
shared = invoice_check.loc[invoice_check["source_sheets"].gt(1)]
conflicts = shared.loc[
    shared["distinct_timestamps"].gt(1)
    | shared["distinct_customers"].gt(1)
    | shared["distinct_countries"].gt(1)
]

invoice_results = pd.Series({
    "Total distinct invoice IDs": len(invoice_check),
    "Shared invoice IDs": len(shared),
    "Shared invoice IDs with date/customer/country differences": len(conflicts),
    "All invoice IDs with multiple customer IDs": int(invoice_check["distinct_customers"].gt(1).sum()),
})
display(invoice_results.to_frame("count"))
display(conflicts.head(15))

## 4. Transaction quality flags

Flag missing values, cancellations, unusual stock codes, and candidate sales without removing rows.


In [ ]:
df["is_cancellation_code"] = df["invoice_id"].str.upper().str.startswith("C", na=False)
df["negative_quantity"] = df["Quantity"].lt(0)
df["zero_quantity"] = df["Quantity"].eq(0)
df["nonpositive_price"] = df["Price"].le(0)
df["missing_customer"] = df["customer_id"].isna()
df["missing_description"] = df["Description"].isna()
df["invalid_timestamp"] = df["invoice_timestamp"].isna()
# Flag uncommon stock-code formats for review; they are not automatically invalid products.
df["unusual_stock_code"] = ~df["stock_code"].str.fullmatch(r"[0-9]{5}[A-Z]?", na=False)

df["positive_sale_candidate"] = (
    ~df["is_cancellation_code"]
    & df["Quantity"].gt(0)
    & df["Price"].gt(0)
    & ~df["invalid_timestamp"]
)
df["customer_sale_candidate"] = df["positive_sale_candidate"] & ~df["missing_customer"]

flags = [
    "cross_sheet_duplicate_candidate", "within_sheet_duplicate",
    "is_cancellation_code", "negative_quantity", "zero_quantity",
    "nonpositive_price", "missing_customer", "missing_description",
    "invalid_timestamp", "unusual_stock_code", "positive_sale_candidate",
    "customer_sale_candidate",
]
quality = pd.DataFrame({
    "rows": df[flags].sum().astype(int),
    "percent": (df[flags].mean() * 100).round(2),
})
display(quality)

## 5. Cancellations, adjustments, and stock-code anomalies

Inspect negative quantities, cancellation codes, and nonstandard stock codes before defining cleaning rules.


In [ ]:
display(pd.crosstab(
    df["is_cancellation_code"],
    df["negative_quantity"],
    rownames=["Cancellation-coded invoice"],
    colnames=["Negative quantity"],
    margins=True,
))

code_review = (
    df.loc[df["unusual_stock_code"]]
      .groupby("stock_code", dropna=False)
      .agg(lines=("StockCode", "size"),
           descriptions=("Description", "nunique"),
           signed_value=("line_amount", "sum"))
      .sort_values("lines", ascending=False)
)
display(code_review.head(25))

display(df.loc[
    df["nonpositive_price"] | df["negative_quantity"],
    ["source_sheet", "invoice_id", "stock_code", "Description",
     "Quantity", "Price", "customer_id", "invoice_timestamp"]
].head(20))

## 6. Candidate sales and validation

Summarize provisional sales subsets and verify basic quality constraints.


In [ ]:
# Candidate sets are analytical views, not final cleaned tables.
sales_candidates = df.loc[df["positive_sale_candidate"]].copy()
identified_sales_candidates = df.loc[df["customer_sale_candidate"]].copy()

candidate_summary = pd.Series({
    "Source lines": len(df),
    "Positive sale candidate lines": len(sales_candidates),
    "Customer-linked sale candidate lines": len(identified_sales_candidates),
    "Candidate lines flagged as cross-sheet duplicates": int(
        sales_candidates["cross_sheet_duplicate_candidate"].sum()
    ),
    "Candidate lines with unusual stock codes": int(
        sales_candidates["unusual_stock_code"].sum()
    ),
})
display(candidate_summary.to_frame("count"))

assert df["source_record_key"].is_unique
assert not sales_candidates["is_cancellation_code"].any()
assert sales_candidates["Quantity"].gt(0).all()
assert sales_candidates["Price"].gt(0).all()
assert not sales_candidates["invalid_timestamp"].any()
assert identified_sales_candidates["customer_id"].notna().all()

print("Validation checks passed. No source records were deleted.")